# Iranian Azerbaijani (azb) — Tokenisation and Translation

Iranian Azerbaijani (Other branch, Arabic script) currently has rule-based tokenisation and prototype-quality morphological analysis. NLLB-200 provides cross-lingual embeddings and machine translation. Iranian Azerbaijani is written in Perso-Arabic script.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp import Pipeline

## 1. Tokenisation

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.detector import detect_script
from turkicnlp.scripts.transliterator import Transliterator

# Iranian Azerbaijani Perso-Arabic text
arab = "من مکتبه گئدیرم."
print("Script Detection:")
print(f"  Detected: {detect_script(arab).name}")
print()

# Perso-Arabic -> Turkic Common Alphabet (Latin)
try:
    t = Transliterator("azb", source=Script.ARABIC, target=Script.COMMON_TURKIC)
    common = t.transliterate(arab)
    print(f"Perso-Arabic:    {arab}")
    print(f"Turkic Common:   {common}")
    
    # Reverse: Turkic Common -> Perso-Arabic
    t_back = Transliterator("azb", source=Script.COMMON_TURKIC, target=Script.ARABIC)
    arab_restored = t_back.transliterate(common)
    print(f"Restored:        {arab_restored}")
    print(f"Round-trip match: {arab == arab_restored}")
except Exception as e:
    print(f"⚠ Note: Transliteration to Turkic Common Alphabet (Latin) may not be fully supported for Iranian Azerbaijani: {e}")
    print("  For Arabic-based languages, use Script.LATIN as alternative")

## 2. Script Detection and Perso-Arabic ↔ Latin Transliteration

Iranian Azerbaijani is written in Perso-Arabic script. The transliteration system enables conversion to Latin for processing.

In [ ]:
turkicnlp.download("azb")
nlp_tok = Pipeline("azb", processors=["tokenize"])
doc = nlp_tok("من مکتبه گئدیرم.")
print([w.text for w in doc.words])

## 2. Morphological Analysis (Apertium FST — Prototype)

In [ ]:
nlp = Pipeline(
    "azb",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
)
doc = nlp("من مکتبه گئدیرم.")
for w in doc.words:
    print(f"{w.text:<18} lemma={w.lemma} feats={w.feats}")

## 3. Translation via NLLB-200

In [ ]:
turkicnlp.download("azb", processors=["translate"])
trans = Pipeline("azb", processors=["translate"], translate_tgt_lang="eng_Latn")
doc = trans("من مکتبه گئدیرم.")
print("EN:", doc.translation)